####Ingestion del archivo "person.json"

####Paso 1 - Leer el archivo JSON usando "DataFrameReader" de Spark

In [0]:
dbutils.widgets.text("p_file_date", "")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%run "../includes/common_functions"

In [0]:
from pyspark.sql.types import StructField, StructType, StringType, IntegerType

In [0]:
name_schema = StructType(fields=[
    StructField("forename", StringType(), True),
    StructField("surname", StringType(), True)
])

In [0]:
persons_schema = StructType(fields=[
    StructField("personId", IntegerType(), False),
    StructField("personName", name_schema)
])

In [0]:
persons_df = spark.read \
    .schema(persons_schema) \
    .json(f"abfss://bronze@moviehistory8.dfs.core.windows.net/{v_file_date}/person.json")

In [0]:
persons_df.printSchema()

root
 |-- personId: integer (nullable = true)
 |-- personName: struct (nullable = true)
 |    |-- forename: string (nullable = true)
 |    |-- surname: string (nullable = true)



In [0]:
display(persons_df)

personId,personName
1617585,"List(David , Martu00ed)"
1617586,"List(Eva , Muslera)"
1617587,"List(Tino , Torrescusa)"
1617588,"List(Yolanda , Cu00e1ceres)"
1617589,"List(Miguel , u00c1ngel Cano)"
1617591,"List(Mar , Francolu00ed)"
1617592,"List(Josu00e9 , Miguel Mendiola)"
1617593,"List(Rafael , Toba)"
1617594,"List(Maru00eda , Steinberg)"
1617595,"List(Gonzalo , Carriu00f3n)"


###Paso 2 - Renombrar las columnas y añadir nuevas columnas
1. "personId" renombrar a "person_id"
2. Agregar las columnas "ingestion_date" y "enviroment"
3. Agregar la columna "name" a partir de la concatenacion de "forename" y "surname"

In [0]:
from pyspark.sql.functions import col, concat, current_timestamp, lit

In [0]:
persons_with_columns_df = persons_df \
                          .withColumnRenamed("personId", "person_id") \
                          .withColumn("ingestion_date", current_timestamp()) \
                          .withColumn("enviroment", lit("Production")) \
                          .withColumn("name",
                                       concat(
                                            col("personName.forename"),
                                            lit(" "),
                                            col("personName.surname")
                                            )
                                    ) \
                          .withColumn("file_date", lit(v_file_date))

In [0]:
display(persons_with_columns_df)

person_id,personName,ingestion_date,enviroment,name,file_date
1617585,"List(David , Martu00ed)",2026-09-26T16:55:51.910384Z,Production,David Martu00ed,2024-12-30
1617586,"List(Eva , Muslera)",2026-09-26T16:55:51.910384Z,Production,Eva Muslera,2024-12-30
1617587,"List(Tino , Torrescusa)",2026-09-26T16:55:51.910384Z,Production,Tino Torrescusa,2024-12-30
1617588,"List(Yolanda , Cu00e1ceres)",2026-09-26T16:55:51.910384Z,Production,Yolanda Cu00e1ceres,2024-12-30
1617589,"List(Miguel , u00c1ngel Cano)",2026-09-26T16:55:51.910384Z,Production,Miguel u00c1ngel Cano,2024-12-30
1617591,"List(Mar , Francolu00ed)",2026-09-26T16:55:51.910384Z,Production,Mar Francolu00ed,2024-12-30
1617592,"List(Josu00e9 , Miguel Mendiola)",2026-09-26T16:55:51.910384Z,Production,Josu00e9 Miguel Mendiola,2024-12-30
1617593,"List(Rafael , Toba)",2026-09-26T16:55:51.910384Z,Production,Rafael Toba,2024-12-30
1617594,"List(Maru00eda , Steinberg)",2026-09-26T16:55:51.910384Z,Production,Maru00eda Steinberg,2024-12-30
1617595,"List(Gonzalo , Carriu00f3n)",2026-09-26T16:55:51.910384Z,Production,Gonzalo Carriu00f3n,2024-12-30


###Paso 3 - Eliminar las columnas no requeridas

In [0]:
persons_final_df = persons_with_columns_df.drop("personName")

In [0]:
display(persons_final_df)

person_id,ingestion_date,enviroment,name,file_date
1617585,2026-09-26T16:55:52.804988Z,Production,David Martu00ed,2024-12-30
1617586,2026-09-26T16:55:52.804988Z,Production,Eva Muslera,2024-12-30
1617587,2026-09-26T16:55:52.804988Z,Production,Tino Torrescusa,2024-12-30
1617588,2026-09-26T16:55:52.804988Z,Production,Yolanda Cu00e1ceres,2024-12-30
1617589,2026-09-26T16:55:52.804988Z,Production,Miguel u00c1ngel Cano,2024-12-30
1617591,2026-09-26T16:55:52.804988Z,Production,Mar Francolu00ed,2024-12-30
1617592,2026-09-26T16:55:52.804988Z,Production,Josu00e9 Miguel Mendiola,2024-12-30
1617593,2026-09-26T16:55:52.804988Z,Production,Rafael Toba,2024-12-30
1617594,2026-09-26T16:55:52.804988Z,Production,Maru00eda Steinberg,2024-12-30
1617595,2026-09-26T16:55:52.804988Z,Production,Gonzalo Carriu00f3n,2024-12-30


###Paso 4 - Escribir la salida en un formato Parquet

In [0]:
#overwrite_partition("movie_silver", "persons", "file_date", v_file_date)

In [0]:
#persons_final_df.write.mode("overwrite").parquet("abfss://silver@moviehistory8.dfs.core.windows.net/persons")

In [0]:
merge_condition = 'tgt.person_id = src.person_id AND tgt.file_date = src.file_date'
merge_delta_lake(persons_final_df, "movie_silver", "persons", merge_condition, "file_date")

In [0]:
#persons_final_df.write.mode("append").partitionBy("file_date").format("delta").saveAsTable("movie_silver.persons")

In [0]:
%sql
SELECT file_date, count(*)
FROM movie_silver.persons
GROUP BY file_date;

file_date,count(1)
2024-12-16,70000
2024-12-23,20000
2024-12-30,14842


In [0]:
%sql
SELECT * FROM movie_silver.persons;

person_id,ingestion_date,enviroment,name,file_date
56716,2026-09-26T16:55:19.500603Z,Production,Cara Silverman,2024-12-16
56717,2026-09-26T16:55:19.500603Z,Production,Tom Vaughan,2024-12-16
56718,2026-09-26T16:55:19.500603Z,Production,Dana Fox,2024-12-16
56719,2026-09-26T16:55:19.500603Z,Production,Willie Chan,2024-12-16
56720,2026-09-26T16:55:19.500603Z,Production,Solon So,2024-12-16
56728,2026-09-26T16:55:19.500603Z,Production,Tim Herlihy,2024-12-16
56730,2026-09-26T16:55:19.500603Z,Production,Cole Sprouse,2024-12-16
56731,2026-09-26T16:55:19.500603Z,Production,Jessica Alba,2024-12-16
56732,2026-09-26T16:55:19.500603Z,Production,Fernanda Romero,2024-12-16
56733,2026-09-26T16:55:19.500603Z,Production,Danny Mora,2024-12-16


In [0]:
#display(spark.read.parquet("abfss://silver@moviehistory8.dfs.core.windows.net/persons"))

In [0]:
dbutils.notebook.exit("Success")